In [32]:
import random
import math
import numpy as np
import pandas as pd

from pathlib import Path

In [33]:
BASE_DIR = Path.cwd().parent

DATA_PATH = BASE_DIR / "data" / "processed" / "priority_prediction_dataset.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,Severity,Disaster_Type,People_Count,Children_Count,Elderly_Count,Injured_Count,Road_Accessibility,Weather_Condition,Location_Type,Time_Since_Incident,Incident_Time,Infrastructure_Damage,Communication_Available,Evacuation_Required,Priority_Score,Priority,Incident_ID
0,0.83,Flood,32,10,3,11,Accessible,Rain,Coastal,74,Morning,Medium,Yes,Yes,40.88,Medium,INC-000001
1,0.89,Flood,84,22,24,21,Not Accessible,Thunderstorm,Urban,85,Morning,High,No,No,71.88,High,INC-000002
2,0.80,Earthquake,177,17,25,91,Not Accessible,Clear,Mountain,20,Evening,Medium,No,Yes,79.48,Critical,INC-000003
3,0.62,Flood,41,18,16,11,Not Accessible,Heavy Rain,Urban,37,Morning,Medium,No,Yes,49.68,Medium,INC-000004
4,0.65,Fire,41,0,9,11,Not Accessible,Rain,Forest,32,Afternoon,Low,Yes,Yes,39.37,Medium,INC-000005


In [34]:
resource_df = df.copy()

print(resource_df.shape)

(50000, 17)


In [36]:
import random
import pandas as pd

def allocate_disaster_resources(row):
    # Unpack row parameters with defaults where applicable
    injured = row.get("injured", 0)
    people = row.get("people", 0)
    children = row.get("children", 0)
    elderly = row.get("elderly", 0)
    vulnerable_people = children + elderly
    
    priority = row.get("priority", "Low")
    severity = row.get("severity", 0.0)
    priority_multiplier = row.get("priority_multiplier", 1.0)
    severity_multiplier = row.get("severity_multiplier", 1.0)
    noise = row.get("noise", 1.0)
    
    road_blocked = row.get("road_blocked", False)
    communication = row.get("communication", "Yes")
    disaster = row.get("disaster", "Unknown")
    infrastructure = row.get("infrastructure", "Low")
    weather = row.get("weather", "Clear")
    location = row.get("location", "Rural")
    evacuation = row.get("evacuation", "No")
    time_since = row.get("time_since", 0) # in minutes
    support_days = row.get("support_days", 3)

    # =====================================================
    # MEDICAL RESOURCE ALLOCATION
    # =====================================================
    ambulances = max(
        1,
        round((injured / 4) * priority_multiplier * severity_multiplier * noise)
    ) + round(vulnerable_people / 40)

    doctors = max(
        1,
        round((injured / 8) * priority_multiplier * severity_multiplier * noise)
    ) + round(vulnerable_people / 25)

    nurses = max(
        doctors,
        round(doctors * 2 * random.uniform(0.9, 1.2))
    ) + round(vulnerable_people / 15)

    medical_kits = max(
        injured,
        round((injured * 1.5 + children * 0.3 + elderly * 0.3) * noise)
    )

    mobile_medical_units = max(1, round((doctors + ambulances) / 8))
    if priority == "Critical":
        mobile_medical_units += 1

    # =====================================================
    # RESCUE OPERATIONS
    # =====================================================
    rescue_teams = max(
        1,
        round((people / 30 + injured / 20) * priority_multiplier * severity_multiplier * noise)
    )
    if road_blocked:
        rescue_teams += 2
    if communication == "No":
        rescue_teams += 2
    if priority == "Critical":
        rescue_teams += 2
    if severity > 0.90:
        rescue_teams += 2

    # =====================================================
    # VEHICLES
    # =====================================================
    fire_trucks = 0
    if disaster == "Fire":
        fire_trucks = min(5, max(2, round((severity * 5 + people / 40) * random.uniform(0.9, 1.1))))
    elif disaster == "Earthquake":
        if infrastructure == "High":
            fire_trucks = random.randint(2, 5)
        elif infrastructure == "Medium":
            fire_trucks = random.randint(1, 3)

    rescue_boats = 0
    if disaster in ["Flood", "Cyclone"]:
        boats = people / 35
        if severity > 0.80:
            boats *= 1.25
        if road_blocked:
            boats *= 1.20
        if weather in ["Heavy Rain", "Cyclone"]:
            boats *= 1.15
        rescue_boats = min(6, max(1, round(boats * noise)))

    rescue_drones = 0
    if disaster in ["Flood", "Cyclone", "Earthquake", "Landslide"]:
        drone_score = (people / 50 + injured / 25 + severity * 2)
        rescue_drones = min(8, max(1, round(drone_score * random.uniform(0.9, 1.1))))

    helicopters = 0
    if priority == "Critical" and road_blocked and severity >= 0.85:
        helicopters = random.randint(2, 4)
    elif disaster == "Landslide" and location == "Mountain":
        helicopters = random.randint(1, 3)
    elif disaster == "Cyclone" and road_blocked:
        helicopters = random.randint(1, 2)

    # =====================================================
    # SECURITY & HUMAN RESOURCES
    # =====================================================
    police = max(2, round((people / 20) * priority_multiplier * severity_multiplier * noise))
    if evacuation == "Yes":
        police += 4
    if location == "Urban":
        police += 3
    if priority == "Critical":
        police += 5
    if communication == "No":
        police += 2

    volunteers = max(
        5,
        round((people / 5 + children / 3 + elderly / 3) * priority_multiplier * severity_multiplier * noise)
    )
    if communication == "No":
        volunteers += 5
    if time_since > 60:
        volunteers += round(volunteers * 0.10)
    if time_since > 120:
        volunteers += round(volunteers * 0.20)

    # =====================================================
    # FOOD & WATER
    # =====================================================
    food_packets = round((people + children * 0.5 + elderly * 0.5) * 3 * support_days * random.uniform(0.95, 1.05))
    water_bottles = round((people * 4 + injured) * support_days * random.uniform(0.95, 1.05))

    if time_since > 120:
        food_packets = round(food_packets * 1.25)
        water_bottles = round(water_bottles * 1.25)
    elif time_since > 60:
        food_packets = round(food_packets * 1.10)
        water_bottles = round(water_bottles * 1.10)

    # =====================================================
    # TEMPORARY SHELTERS
    # =====================================================
    shelters = 0
    if evacuation == "Yes":
        shelters = max(1, round((people / 45) * noise))
        food_packets += shelters * 20
        water_bottles += shelters * 50

    supply_trucks = max(1, round((food_packets + water_bottles) / 1200))

    # =====================================================
    # HEAVY MACHINERY
    # =====================================================
    bulldozers = 0
    if disaster in ["Earthquake", "Landslide"]:
        bulldozers = max(1, round((people / 80 + severity * 2) * noise))

    excavators = 0
    if disaster == "Earthquake":
        excavators = max(1, round((injured / 30 + severity) * noise))
    elif disaster == "Landslide":
        excavators = max(1, round((people / 60 + severity) * noise))

    cranes = 0
    if disaster == "Earthquake":
        if infrastructure == "High":
            cranes = random.randint(2, 5)
        elif infrastructure == "Medium":
            cranes = random.randint(1, 3)

    # =====================================================
    # INFRASTRUCTURE SUPPORT
    # =====================================================
    power_generators = 0
    if infrastructure == "High":
        power_generators = max(2, round(people / 40))
    elif infrastructure == "Medium":
        power_generators = max(1, round(people / 80))

    if communication == "No":
        power_generators += 2

    search_dogs = 0
    if disaster in ["Earthquake", "Landslide"]:
        search_dogs = max(2, round(injured / 20))

    communication_units = 0
    if communication == "No":
        communication_units = random.randint(2, 8)

    fuel_tankers = max(1, round((ambulances + fire_trucks + rescue_boats + helicopters) / 8))

    # =====================================================
    # ESTIMATED DEPLOYMENT COST
    # =====================================================
    estimated_deployment_cost = (
        ambulances * 12000 +
        doctors * 5000 +
        nurses * 3000 +
        fire_trucks * 300000 +
        rescue_boats * 500000 +
        helicopters * 5000000 +
        police * 2500 +
        volunteers * 800 +
        food_packets * 150 +
        water_bottles * 20 +
        bulldozers * 150000 +
        excavators * 180000 +
        cranes * 250000 +
        power_generators * 80000 +
        mobile_medical_units * 1000000 +
        rescue_drones * 120000 +
        search_dogs * 50000 +
        communication_units * 150000 +
        fuel_tankers * 100000
    )

    # =====================================================
    # RETURN RESOURCE ALLOCATION
    # =====================================================
    return pd.Series({
        "Ambulances": ambulances,
        "Doctors": doctors,
        "Nurses": nurses,
        "Medical_Kits": medical_kits,
        "Mobile_Medical_Units": mobile_medical_units,
        "Rescue_Teams": rescue_teams,
        "Fire_Trucks": fire_trucks,
        "Rescue_Boats": rescue_boats,
        "Helicopters": helicopters,
        "Rescue_Drones": rescue_drones,
        "Police": police,
        "Volunteers": volunteers,
        "Food_Packets": food_packets,
        "Water_Bottles": water_bottles,
        "Temporary_Shelters": shelters,
        "Supply_Trucks": supply_trucks,
        "Bulldozers": bulldozers,
        "Excavators": excavators,
        "Cranes": cranes,
        "Power_Generators": power_generators,
        "Communication_Units": communication_units,
        "Fuel_Tankers": fuel_tankers,
        "Search_Dogs": search_dogs,
        "Estimated_Deployment_Cost": round(estimated_deployment_cost, 2)
    })

In [37]:
resource_columns = resource_df.apply(
    allocate_resources,
    axis=1
)

In [38]:
resource_df = pd.concat(
    [resource_df, resource_columns],
    axis=1
)

resource_df.head()

,Severity,Disaster_Type,People_Count,Children_Count,Elderly_Count,Injured_Count,Road_Accessibility,Weather_Condition,Location_Type,Time_Since_Incident,...,Water_Bottles,Temporary_Shelters,Supply_Trucks,Bulldozers,Excavators,Cranes,Power_Generators,Search_Dogs,Communication_Units,Fuel_Tankers
0,0.83,Flood,32,10,3,11,Accessible,Rain,Coastal,74,...,365,1,1,0,0,0,2,0,0,1
1,0.89,Flood,84,22,24,21,Not Accessible,Thunderstorm,Urban,85,...,1415,0,1,0,0,0,5,0,7,2
2,0.80,Earthquake,177,17,25,91,Not Accessible,Clear,Mountain,20,...,5009,4,4,2,3,0,6,5,4,5
3,0.62,Flood,41,18,16,11,Not Accessible,Heavy Rain,Urban,37,...,424,1,1,0,0,0,3,0,4,1
4,0.65,Fire,41,0,9,11,Not Accessible,Rain,Forest,32,...,453,1,1,0,0,0,0,0,0,1


In [39]:
print(resource_df.shape)

(50000, 40)


In [40]:
print(resource_df.isnull().sum())

Severity                   0
Disaster_Type              0
People_Count               0
Children_Count             0
Elderly_Count              0
Injured_Count              0
Road_Accessibility         0
Weather_Condition          0
Location_Type              0
Time_Since_Incident        0
Incident_Time              0
Infrastructure_Damage      0
Communication_Available    0
Evacuation_Required        0
Priority_Score             0
Priority                   0
Incident_ID                0
Ambulances                 0
Doctors                    0
Nurses                     0
Medical_Kits               0
Mobile_Medical_Units       0
Rescue_Teams               0
Fire_Trucks                0
Rescue_Boats               0
Helicopters                0
Rescue_Drones              0
Police                     0
Volunteers                 0
Food_Packets               0
Water_Bottles              0
Temporary_Shelters         0
Supply_Trucks              0
Bulldozers                 0
Excavators    

In [41]:
print("Duplicate Rows:", resource_df.duplicated().sum())

Duplicate Rows: 0


In [42]:
resource_df[
    [
        "Priority",
        "Ambulances",
        "Doctors",
        "Nurses",
        "Medical_Kits",
        "Mobile_Medical_Units",
        "Rescue_Teams",
        "Fire_Trucks",
        "Rescue_Boats",
        "Helicopters",
        "Rescue_Drones",
        "Police",
        "Volunteers",
        "Food_Packets",
        "Water_Bottles",
        "Temporary_Shelters",
        "Supply_Trucks",
        "Bulldozers",
        "Excavators",
        "Cranes",
        "Power_Generators",
        "Search_Dogs",
        "Communication_Units",
        "Fuel_Tankers"
    ]
].head(10)

,Priority,Ambulances,Doctors,Nurses,Medical_Kits,Mobile_Medical_Units,Rescue_Teams,Fire_Trucks,Rescue_Boats,Helicopters,...,Water_Bottles,Temporary_Shelters,Supply_Trucks,Bulldozers,Excavators,Cranes,Power_Generators,Search_Dogs,Communication_Units,Fuel_Tankers
0,Medium,3,3,7,20,1,2,0,2,0,...,365,1,1,0,0,0,2,0,0,1
1,High,8,6,17,43,1,8,0,5,0,...,1415,0,1,0,0,0,5,0,7,2
2,Critical,41,22,46,152,5,23,1,0,0,...,5009,4,4,2,3,0,6,5,4,5
3,Medium,4,2,6,26,1,5,0,4,1,...,424,1,1,0,0,0,3,0,4,1
4,Medium,3,2,5,20,1,4,7,0,0,...,453,1,1,0,0,0,0,0,0,1
5,High,21,11,22,90,2,13,0,0,0,...,1795,3,1,1,2,0,2,3,4,3
6,Medium,2,1,2,10,1,1,7,0,0,...,114,1,1,0,0,0,8,0,0,1
7,High,9,6,12,46,1,12,0,11,0,...,2153,3,2,0,0,0,4,0,2,2
8,High,2,1,3,10,1,4,0,0,3,...,261,1,1,1,1,0,2,2,5,1
9,High,6,3,9,29,1,7,0,9,0,...,2336,3,2,0,0,0,2,0,0,2


In [43]:
from pathlib import Path

BASE_DIR = Path.cwd().parent

SAVE_PATH = (
    BASE_DIR
    / "data"
    / "processed"
    / "resource_allocation_dataset.csv"
)

resource_df.to_csv(
    SAVE_PATH,
    index=False
)

print("Dataset saved successfully!")
print(SAVE_PATH)

Dataset saved successfully!
C:\Users\HP\n1\ResqAI\data\processed\resource_allocation_dataset.csv
